# Layer 3 · Step 1: the baselines, no generator yet

Step 1 of `Essential_task_delegation/Layer3_Task/Layer3_Task_Brief.md`, built 18 Nov 2026. **GPU T4** (one is used). **Internet on**, to install segmentation-models-pytorch 0.3.4, as E5 step 2 did.

**What it does, and why**
- It runs R11 and R9 on the 645 validation pairs exactly as `Champion_Task/e5-step2-per-image-counts.ipynb` did:
  - full size, padded to a multiple of 32;
  - 8-view TTA;
  - the same metric code.

  It saves each pair's 16 pixel counts (true class against predicted class).
- It changes one thing: `BEFORE_SOURCE`, the picture R11 reads in its before-photo slot. R11 runs once for each source listed:
  - `real`: the real before-photo;
  - `copy`: the after-photo, copied into the slot;
  - `gray`: a flat 1024 × 1024 RGB picture in the mean colour of the 5,798 training before-photos (Abrar, 18 Nov). Cell 5 computes it: (R, G, B) = (86, 91, 69);
  - `folder:<path>`: generated pictures from a folder, named like the after-photos (used by steps 2b, 3 and 5).

  Every source goes through the same normalisation as the real before-photo.
- R9 reads only the after-photo, so it runs once.
- It reads step 0's `pairs.csv` for the 605 clean pairs, and checks that the split it rebuilds is the same.
- **Why:**
  - `real` and R9 must reproduce E5 step 2. Otherwise nothing after this can be compared.
  - `copy` and `gray` show what R11 does with a before-photo that carries no information.
  - The pass rule's third test compares a generator with `copy`.

**The checks.** The last cell prints all five. If any fails, it stops with an error, and the printout goes to Abrar.
1. `pairs.csv` comes from step 0's full run, and the rebuilt split matches it pair for pair:
   - 6,443 pairs: 5,798 training and 645 validation;
   - 605 clean.
2. The gray colour: the 5,798 training before-photos sum to exactly (R, G, B) = (520,038,244,565, 551,716,716,020, 421,116,766,778) over 6,079,643,648 pixels. That is a mean of (85.5376, 90.7482, 69.2667), rounded to (86, 91, 69).
3. R11 with the real before-photo reproduces E5 step 2 within 0.001, with 8-view TTA:
   - on the 605 clean pairs: mIoU 0.5805, destroyed IoU 0.2692;
   - on all 645: 0.5875 and 0.2965.
4. R9 reproduces E5 step 2 within 0.001:
   - on the 605 clean pairs: 0.5402 and 0.2874;
   - on all 645: 0.5440 and 0.3001.
5. Every run scores every validation pair.

**In the smoke test, checks 3 and 4 use its 8 pairs** (E5 step 2's first 8), against E5's counts for them:
- R11: 0.6190 and 0.0906;
- R9: 0.5546 and 0.1227.

The tolerance there is 0.005. On 8 pairs the destroyed IoU rests on about 12,800 pixels, so a dozen pixels that come out differently on another GPU move it by 0.001.

**The expectation, fixed on 18 Nov before any run**
- All five checks pass. The bar for step 2 then stays **0.5604** = 0.5402 + (0.5805 − 0.5402) / 2.
- From E5 step 2's counts, on the 605 clean pairs with TTA:
  - building IoU: R11 real 0.7429, R9 0.5905;
  - building precision: R11 real 0.8347, R9 0.6687;
  - R11 real minus R9 in mIoU: 0.0403. Over 10,000 resamples of the pairs (seed 42), the middle 95% runs from 0.0154 to 0.0632.
- `copy` and `gray` have no earlier numbers.
  - We expect both below R11 real, because they carry no before-photo information.
  - They are probably below R9 too, because R11 never saw such a before-photo in training.
  - Neither should pass the rule. If one does, that's a surprise to report.

**How to run on Kaggle**
1. Settings: Accelerator **GPU T4 x2** (or T4), Internet **on**.
2. Inputs (+ Add Input):
   - the dataset `abrarmohammedtanzim/disasterm3-mirror`;
   - the dataset `abrarmohammedtanzim/disasterm3-r11-r9-checkpoints`, shared with our Kaggle group (no Hugging Face token is needed);
   - your step 0 notebook, `layer3-step0-pairs-and-checks`: the output of its full run.
3. Smoke test: Run All with `SMOKE_TEST = True` (8 pairs per run). If Cell 1 says RESTART THE KERNEL NOW, restart (Run → Restart & clear cell outputs) and Run All again. It takes about 30 minutes: Cell 3 alone, which builds the 6,443 masks, took 16 to 22 minutes in step 0's Kaggle runs.
4. After Abrar's go: set `SMOKE_TEST = False`, then Save Version → Save & Run All (Commit). It takes about 1 hour.
5. Outputs in `/kaggle/working/layer3-step1/`: `per_pair.csv`, `scores.csv`, `rule_tests.csv` and `summary.json` (section 3 says what each holds).

In [1]:
# ── Cell 1: E5 step 2's Cell 1, word for word ──
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed to load the D4 checkpoints (their key layout).
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu130, python 3.13.16


In [2]:
# ── Cell 2: Configuration and inputs (E5 step 2's Cell 2; changed: BEFORE_SOURCE, R11 and R9 only, step 0's pairs.csv) ──
import glob
import hashlib
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
MASK_TMP = "/tmp/combined_masks"    # where R11's cells write the 4-class masks

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
SMOKE_TEST = False                  # True = 8 validation pairs per run (about 30 min); False = all 645 (about 1 hour)
OUT_DIR = Path("/kaggle/working/layer3-step1")
HF_USER = "AbrarAlam"               # E5's fallback only: here the checkpoints come from Abrar's Kaggle dataset

# ── The one change: the picture in R11's before-photo slot. R11 runs once for each entry. ──
#   "real": the real before-photo; "copy": the after-photo; "gray": the mean colour of the training before-photos
#   (Cell 5); "folder:<path>": generated pictures named like the after-photos (steps 2b, 3 and 5)
BEFORE_SOURCE = ["real", "copy", "gray"]

# in_channels: 6 = after-image then before-image (R11's recipe), 3 = after-image only.
# expected: (mIoU, Destroyed IoU) with 8-view TTA, from E5 step 2: on the 605 clean pairs (the brief), on all 645
# (its reproduction.csv) and on the smoke test's 8 pairs (its per_image_counts.csv)
MODELS = {
    "R11": {"in_channels": 6, "expected": {"605": (0.5805, 0.2692), "645": (0.5875, 0.2965), "8": (0.6190, 0.0906)}},
    "R9":  {"in_channels": 3, "expected": {"605": (0.5402, 0.2874), "645": (0.5440, 0.3001), "8": (0.5546, 0.1227)}},
}
RUN_MODELS = ["R11", "R9"]
TOLERANCE = {"605": 0.001, "645": 0.001, "8": 0.005}   # the brief: 0.001; the smoke test's 8 pairs: 0.005 (Cell 0)
E5_BAR = 0.5604                     # 0.5402 + (0.5805 - 0.5402) / 2: kept if R11 real and R9 reproduce E5 step 2
GRAY_EXPECTED = {"sums": [520038244565, 551716716020, 421116766778], "pixels": 6079643648, "rgb": [86, 91, 69]}
SEED = 42                           # the brief: seed 42 everywhere (E7 step 2b's resampling used 0)
RESAMPLES = 10_000                  # the rule: the 605 pairs redrawn with replacement 10,000 times

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add the HF_TOKEN secret, or a Kaggle dataset with {name}/final_model.pth.")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ── The three inputs: the dataset, Abrar's checkpoints and step 0's output ──
if not MANIFEST_PATH.exists():      # the mount path can differ between accounts: look for the manifest
    _found = sorted(glob.glob("/kaggle/input/**/DisasterM3_Instruct/train_release.json", recursive=True))
    assert _found, "No manifest: attach the dataset abrarmohammedtanzim/disasterm3-mirror"
    MANIFEST_PATH = Path(_found[0])
    DATA_ROOT = MANIFEST_PATH.parent
for _m in RUN_MODELS:
    assert glob.glob(f"/kaggle/input/**/{_m}/final_model.pth", recursive=True), (
        f"No {_m}/final_model.pth: attach abrarmohammedtanzim/disasterm3-r11-r9-checkpoints (no Hugging Face token needed)")
_step0 = sorted(glob.glob("/kaggle/input/**/layer3-step0/pairs.csv", recursive=True))
assert _step0, "No pairs.csv: attach the output of your step 0 notebook (layer3-step0-pairs-and-checks, its full run)"
STEP0_DIR = Path(_step0[0]).parent
assert "real" in BEFORE_SOURCE and "copy" in BEFORE_SOURCE, "keep real (the reproduction check) and copy (the rule's third test)"
for _s in BEFORE_SOURCE:
    assert _s in ("real", "copy", "gray") or (_s.startswith("folder:") and Path(_s[len("folder:"):]).is_dir()), (
        f"BEFORE_SOURCE {_s!r}: use real, copy, gray or folder:<an existing folder>")

CKPTS = {m: find_checkpoint(m) for m in RUN_MODELS}
CKPT_SHA256 = {m: sha256(CKPTS[m]) for m in RUN_MODELS}
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T_START = time.time()
print(f"✓ Layer 3 step 1{'  [SMOKE TEST]' if SMOKE_TEST else ''}: R11 with each before-photo in {BEFORE_SOURCE}, and R9")
print(f"  Data: {DATA_ROOT}")
print(f"  Step 0's pairs: {STEP0_DIR / 'pairs.csv'}")
for m in RUN_MODELS:
    print(f"  {m} ({MODELS[m]['in_channels']} channels): {CKPTS[m]}")
    print(f"      SHA-256 {CKPT_SHA256[m]}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")

✓ Layer 3 step 1: R11 with each before-photo in ['real', 'copy', 'gray'], and R9
  Data: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct
  Step 0's pairs: /kaggle/input/notebooks/riditaafrin/layer3-step0-pairs-and-checks/layer3-step0/pairs.csv
  R11 (6 channels): /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-r11-r9-checkpoints/R11/final_model.pth
      SHA-256 2ae816576ad25e6bd61ca9322d205524ec24cb3d63f5f15b464665d2172673c0
  R9 (3 channels): /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-r11-r9-checkpoints/R9/final_model.pth
      SHA-256 e1f16b716cc3c50e2d99d7d656962e1a606642ff17339ffb36a7e44d11735836
  Device: Tesla T4; outputs: /kaggle/working/layer3-step1


## 1. The data: the same 645 validation pairs as every D4 run, checked against step 0
- Cell 3 is E5 step 2's Cell 3, word for word. That is R11's Cell 4, except for its mask folder. It builds the 6,443 pairs and their 4-class masks.
- Cell 4 is E5 step 2's split, word for word, plus one line that keeps all 645 validation pairs for the comparison. It then checks the split against step 0's `pairs.csv`, which also gives each pair's clean flag.
- Cell 5 makes the gray picture from the 5,798 training before-photos.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (R11's Cell 4, verbatim but for MASK_TMP) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs(MASK_TMP, exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"{MASK_TMP}/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)


In [4]:
# ── Cell 4: The validation split and event names (E5 step 2's Cell 4, word for word but for one added line), then step 0's pairs.csv ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
VAL_ALL = val_pairs                  # all 645 validation pairs, compared with step 0's pairs.csv below
if SMOKE_TEST:
    val_pairs = val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in val_pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation: {len(val_pairs)} images ({n_xbd} xBD, {len(val_pairs) - n_xbd} other). "
      "Expected 645 (560 xBD, 85 other) for the full run.")
if not SMOKE_TEST:
    assert (len(val_pairs), n_xbd) == (645, 560), "not the D4 split"

# ── Added for Layer 3: step 0's pairs.csv must hold this same split. It also gives each pair's clean flag. ──
import pandas as pd

P0 = pd.read_csv(STEP0_DIR / "pairs.csv")
S0 = json.loads((STEP0_DIR / "summary.json").read_text(encoding="utf-8"))

def _rows(split):
    p = P0[P0["split"] == split]
    return set(zip(p["after_image"], p["before_image"]))

def _names(ps):
    return {(Path(p["image_path"]).name, Path(p["pre_image_path"]).name) for p in ps}

_val0 = P0[P0["split"] == "val"]
VAL_CLEAN = dict(zip(_val0["after_image"], _val0["clean"].eq(True)))
_events0 = dict(zip(_val0["after_image"], zip(_val0["event"], _val0["source"])))
STEP0_FINDINGS = {
    "step 0's full run, every check passed": not S0["smoke_test"] and all(v == "passed" for v in S0["checks"].values()),
    "6,443 pairs": len(P0) == len(pairs) == 6443,
    "the same 5,798 training pairs": len(train_pairs) == 5798 and _rows("train") == _names(train_pairs),
    "the same 645 validation pairs": len(VAL_ALL) == 645 and _rows("val") == _names(VAL_ALL),
    "605 clean": sum(VAL_CLEAN.values()) == 605,
    "the same events": all(_events0.get(Path(p["image_path"]).name) == event_and_source(p["image_path"]) for p in VAL_ALL),
}
SAME_AS_STEP0 = all(STEP0_FINDINGS.values())
n_clean = sum(VAL_CLEAN[Path(p["image_path"]).name] for p in val_pairs)
print(f"{'✓' if SAME_AS_STEP0 else '✗'} Step 0's pairs.csv: "
      + "; ".join(f"{k}: {'yes' if v else 'NO'}" for k, v in STEP0_FINDINGS.items()))
print(f"  {n_clean} of these {len(val_pairs)} validation pairs are clean")

✓ Validation: 645 images (560 xBD, 85 other). Expected 645 (560 xBD, 85 other) for the full run.
✓ Step 0's pairs.csv: step 0's full run, every check passed: yes; 6,443 pairs: yes; the same 5,798 training pairs: yes; the same 645 validation pairs: yes; 605 clean: yes; the same events: yes
  605 of these 645 validation pairs are clean


In [5]:
# ── Cell 5: The gray picture: the mean colour of the 5,798 training before-photos (Abrar, 18 Nov) ──
from concurrent.futures import ThreadPoolExecutor

def channel_sums(path):
    """Each colour's sum over one photo, read as read_triplet (Cell 6) reads it: cv2, then BGR to RGB."""
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return im.reshape(-1, 3).sum(0, dtype=np.int64), im.shape

t0 = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
    _sums = list(ex.map(channel_sums, [p["pre_image_path"] for p in train_pairs]))
GRAY_PHOTOS = len(_sums)
GRAY_SUMS = [int(v) for v in np.sum([s for s, _ in _sums], axis=0)]
GRAY_PIXELS = sum(shape[0] * shape[1] for _, shape in _sums)
GRAY_MEAN = [round(s / GRAY_PIXELS, 4) for s in GRAY_SUMS]
GRAY_RGB = [int(v) for v in np.round(np.array(GRAY_SUMS) / GRAY_PIXELS)]
GRAY = np.full((1024, 1024, 3), GRAY_RGB, np.uint8)     # Abrar: a flat 1024 x 1024 RGB uint8 picture of the mean colour
GRAY_OK = (GRAY_PHOTOS, GRAY_SUMS, GRAY_PIXELS, GRAY_RGB) == (
    5798, GRAY_EXPECTED["sums"], GRAY_EXPECTED["pixels"], GRAY_EXPECTED["rgb"])
_sizes = Counter(f"{shape[1]}x{shape[0]}" for _, shape in _sums)
print(f"{'✓' if GRAY_OK else '✗'} Gray picture: the mean (R, G, B) of {GRAY_PHOTOS:,} training before-photos "
      f"({dict(_sizes)}) is {GRAY_MEAN}, so {GRAY_RGB} as uint8: "
      f"{'as expected' if GRAY_OK else 'NOT as expected ' + str(GRAY_EXPECTED)} ({(time.time() - t0) / 60:.1f} min)")

✓ Gray picture: the mean (R, G, B) of 5,798 training before-photos ({'1024x1024': 5798}) is [85.5376, 90.7482, 69.2667], so [86, 91, 69] as uint8: as expected (4.3 min)


## 2. R11 with each before-photo, and R9
- Cell 6 is E5 step 2's prediction and metric code, word for word. Two functions are added:
  - `before_photo()` fills R11's before-photo slot from `BEFORE_SOURCE`. Its picture then goes through `unet_input()`, like the real before-photo;
  - `four_scores()` adds building precision to E5's metrics.
- Cell 7 is E5 step 2's loop. R11 runs once for each before-photo source, and R9 once.

In [6]:
# ── Cell 6: E5 step 2's Cell 5, word for word; then the before-photo slot and building precision (added) ──
# ── Cell 5: The U-Nets (the D4 prediction code) and the E-protocol metrics ──
import segmentation_models_pytorch as smp
import torch.nn.functional as F

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def load_unet(path, in_channels):
    """D4 checkpoint: U-Net + ResNet-50, 4 classes; 6 input channels (after-image, then before-image) or 3 (after only)."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=in_channels, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre, in_channels):
    """As the D4 val_transform: ImageNet normalisation; channels 0-2 the after-image, 3-5 the before-image (R11 only)."""
    parts = [(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD]
    if in_channels == 6:
        parts.append((pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD)
    x = np.concatenate(parts, 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
assert unet_input(np.zeros((4, 4, 3), np.uint8), np.zeros((4, 4, 3), np.uint8), 3).shape[1] == 3
assert unet_input(np.zeros((4, 4, 3), np.uint8), np.zeros((4, 4, 3), np.uint8), 6).shape[1] == 6
print("✓ Prediction code ready (self-checks passed)")

# ── Added for Layer 3: what R11 reads in its before-photo slot, and building precision ──
def before_photo(source, pair, post, pre):
    """The picture for R11's before-photo slot. It then goes through unet_input, like the real before-photo."""
    if source == "real":
        return pre
    if source == "copy":
        return post.copy()
    if source == "gray":
        assert post.shape == GRAY.shape, f"the gray picture is 1024 x 1024, but {pair['image_path']} is {post.shape}"
        return GRAY
    if source.startswith("folder:"):
        path = Path(source[len("folder:"):]) / Path(pair["image_path"]).name
        im = cv2.imread(str(path))
        if im is None:
            raise FileNotFoundError(f"No generated picture at {path}")
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        if im.shape != post.shape:     # as read_triplet treats the real before-photo
            im = cv2.resize(im, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
        return im
    raise ValueError(f"Unknown BEFORE_SOURCE: {source!r}")

def four_scores(cm):
    """mIoU, destroyed IoU and building IoU (E5's metrics), and building precision: of the pixels the map calls
    a building (classes 1-3), the share that truly are one."""
    m = metrics(cm)
    return {"miou": m["miou"], "destroyed_iou": float(m["iou"][3]), "building_iou": m["building_iou"],
            "building_precision": float(cm[1:, 1:].sum() / (cm[:, 1:].sum() + 1e-8))}

# Every generated picture must be there before any GPU time is spent
for _s in BEFORE_SOURCE:
    if _s.startswith("folder:"):
        _missing = [Path(p["image_path"]).name for p in val_pairs
                    if not (Path(_s[len("folder:"):]) / Path(p["image_path"]).name).exists()]
        assert not _missing, f"{len(_missing)} generated pictures missing in {_s[len('folder:'):]}, e.g. {_missing[:3]}"

_post = np.random.default_rng(SEED).integers(0, 256, (1024, 1024, 3), dtype=np.uint8)
_pre = np.zeros_like(_post)
assert before_photo("real", {"image_path": "x"}, _post, _pre) is _pre
assert np.array_equal(before_photo("copy", {"image_path": "x"}, _post, _pre), _post)
assert (before_photo("gray", {"image_path": "x"}, _post, _pre).reshape(-1, 3) == GRAY_RGB).all()
assert unet_input(_post, before_photo("gray", {"image_path": "x"}, _post, _pre), 6).shape == (1, 6, 1024, 1024)
_s4 = four_scores(confusion(np.array([[0, 1], [2, 3]], np.uint8), np.array([[0, 1], [1, 0]], np.uint8)))
assert abs(_s4["building_precision"] - 1) < 1e-6 and abs(_s4["building_iou"] - 2 / 3) < 1e-6
print(f"✓ Before-photo slot ready for {BEFORE_SOURCE} (self-checks passed)")

✓ Prediction code ready (self-checks passed)
✓ Before-photo slot ready for ['real', 'copy', 'gray'] (self-checks passed)


In [7]:
# ── Cell 7: Each run on the validation pairs: one 4x4 count per pair and mode (E5 step 2's Cell 6; R11 runs once
#    per before-photo source, and each pair is read once per model) ──
from tqdm.auto import tqdm

rows, model_minutes = [], {}
for name in RUN_MODELS:
    t0 = time.time()
    unet = load_unet(CKPTS[name], MODELS[name]["in_channels"])
    sources = BEFORE_SOURCE if MODELS[name]["in_channels"] == 6 else ["none"]     # R9 has no before-photo slot
    for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc=name):
        data = read_triplet(pair)
        if data is None:
            raise RuntimeError(f"Unreadable validation pair: {pair['image_path']}")
        post, pre, gt = data
        for before in sources:
            slot = pre if before == "none" else before_photo(before, pair, post, pre)
            pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, slot, MODELS[name]["in_channels"]))
            for mode, pred in (("std", pred_std), ("tta", pred_tta)):
                cm = confusion(gt, pred)
                rows.append({"run": name if before == "none" else f"{name} {before}", "model": name,
                             "before_source": before, "mode": mode, "image": Path(pair["image_path"]).name,
                             "before_image": Path(pair["pre_image_path"]).name, "event": event, "source": source,
                             "clean": VAL_CLEAN[Path(pair["image_path"]).name],
                             **{f"t{i}_p{j}": int(cm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)}})
    del unet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model_minutes[name] = round((time.time() - t0) / 60, 1)
    print(f"✓ {name}: {len(val_pairs)} pairs × {len(sources)} before-photo source(s) in {model_minutes[name]} min")

R11:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R11: 645 pairs × 3 before-photo source(s) in 20.5 min


R9:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R9: 645 pairs × 1 before-photo source(s) in 5.8 min


## 3. The results, the checks and the files
Cell 8 writes four files to `/kaggle/working/layer3-step1/`:
- `per_pair.csv`: one row for each run, mode (standard or 8-view TTA) and pair, with the pair's 16 counts `t<truth>_p<prediction>`;
- `scores.csv`: mIoU, destroyed IoU, building IoU and building precision for each run and mode:
  - on all pairs and on the clean ones;
  - overall, for Turkey, for the rest, and for each event;
- `rule_tests.csv`: the rule's three tests for each R11 run on the 605 clean pairs, by the resampling of E7 step 2b (10,000 resamples, seed 42). For `real`, `copy` and `gray` they decide nothing. They are here so that step 2b can run this same notebook with only `BEFORE_SOURCE` changed;
- `summary.json`: the settings, the library versions, the checkpoints' SHA-256, the gray colour, the reproduction, the bar and the checks.

In [8]:
# ── Cell 8: The scores, the reproduction check, the bar, the rule's tests, the output files and the five checks ──
import platform
import sklearn

CM_COLS = [f"t{i}_p{j}" for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)]
per_pair = pd.DataFrame(rows)
per_pair.to_csv(OUT_DIR / "per_pair.csv", index=False)
RUNS = list(dict.fromkeys(per_pair["run"]))          # "R11 real", "R11 copy", "R11 gray", ..., "R9"

def summed(df):
    """The 4 x 4 count summed over these rows' pairs."""
    return df[CM_COLS].to_numpy().sum(0).reshape(NUM_CLASSES, NUM_CLASSES)

# ── scores.csv: each run and mode, on all pairs and on the clean ones: overall, Turkey, the rest, each event ──
score_rows = []
for (run, mode), g in per_pair.groupby(["run", "mode"], sort=False):
    for pairs_name, gp in (("all", g), ("clean", g[g["clean"]])):
        turkey = gp["event"].str.startswith("turkey")
        groups = [("all", gp), ("Turkey", gp[turkey]), ("not Turkey", gp[~turkey])]
        groups += [(f"event: {e}", ge) for e, ge in gp.groupby("event")]
        for group, gg in groups:
            if len(gg):
                score_rows.append({"run": run, "mode": mode, "pairs": pairs_name, "group": group, "n_pairs": len(gg),
                                   **{k: round(v, 6) for k, v in four_scores(summed(gg)).items()}})
scores = pd.DataFrame(score_rows)
scores.to_csv(OUT_DIR / "scores.csv", index=False)

# ── Does R11 real and R9 reproduce E5 step 2? 8-view TTA (smoke test: its 8 pairs) ──
def tta_scores(run, clean_only):
    g = per_pair[(per_pair["run"] == run) & (per_pair["mode"] == "tta")]
    return four_scores(summed(g[g["clean"]] if clean_only else g))

repro = []
for model, run in (("R11", "R11 real"), ("R9", "R9")):
    for key, clean_only in ((("8", False),) if SMOKE_TEST else (("605", True), ("645", False))):
        s, (e_miou, e_dest) = tta_scores(run, clean_only), MODELS[model]["expected"][key]
        repro.append({"run": run, "pairs": key, "miou": round(s["miou"], 4), "destroyed_iou": round(s["destroyed_iou"], 4),
                      "e5_miou": e_miou, "e5_destroyed_iou": e_dest, "tolerance": TOLERANCE[key],
                      "reproduces_e5": abs(s["miou"] - e_miou) <= TOLERANCE[key]
                                       and abs(s["destroyed_iou"] - e_dest) <= TOLERANCE[key]})
repro = pd.DataFrame(repro)
REPRODUCED = bool(repro["reproduces_e5"].all())

# ── The bar for step 2, from our own numbers on the 605 clean pairs. The brief: it stays 0.5604 if both reproduce. ──
OWN = {run: tta_scores(run, True)["miou"] for run in ("R11 real", "R9")}
OWN_BAR = OWN["R9"] + (OWN["R11 real"] - OWN["R9"]) / 2
BAR = E5_BAR if REPRODUCED and not SMOKE_TEST else None

# ── The rule's three tests (brief, section 4) for each R11 run, on the 605 clean pairs (smoke test: its 8 pairs) ──
def tp_union(cms):
    """Per pair and class: the pixels that truth and map both give that class, and the pixels either gives it."""
    tp = np.diagonal(cms, axis1=1, axis2=2).astype(np.float64)
    return tp, cms.sum(2) + cms.sum(1) - tp

def miou_resampled(w, tp, union):
    """w: (resamples, pairs) weights -> each resample's mIoU from its summed counts (as in E7 step 2b's gap())."""
    return ((w @ tp) / (w @ union + 1e-8)).mean(1)

judged = per_pair[(per_pair["mode"] == "tta") & (per_pair["clean"] | SMOKE_TEST)]
order = sorted(judged["image"].unique())
n_judged = len(order)
TPU = {run: tp_union(g.set_index("image").loc[order, CM_COLS].to_numpy().reshape(-1, NUM_CLASSES, NUM_CLASSES))
       for run, g in judged.groupby("run")}
rng = np.random.default_rng(SEED)
W = np.concatenate([rng.multinomial(n_judged, np.full(n_judged, 1 / n_judged), size=1000).astype(np.float64)
                    for _ in range(RESAMPLES // 1000)])     # E7 step 2b's draws; the same 10,000 for every comparison
OBSERVED = {run: float(miou_resampled(np.ones((1, n_judged)), *tu)[0]) for run, tu in TPU.items()}
RESAMPLED = {run: miou_resampled(W, *tu) for run, tu in TPU.items()}

def gain(a, b):
    """mIoU(a) - mIoU(b): observed, and the middle 95% of the resampled differences."""
    lo, hi = np.percentile(RESAMPLED[a] - RESAMPLED[b], [2.5, 97.5])
    return round(OBSERVED[a] - OBSERVED[b], 4), round(float(lo), 4), round(float(hi), 4)

ROLE = {"real": "reference: the real before-photo", "copy": "baseline: test 3 compares with it",
        "gray": "baseline: a flat picture"}
rule_rows = []
for src in BEFORE_SOURCE:
    run = f"R11 {src}"
    r9, cp = gain(run, "R9"), (gain(run, "R11 copy") if src != "copy" else (None, None, None))
    t1 = None if BAR is None else OBSERVED[run] >= BAR
    t2 = r9[1] > 0
    t3 = None if src == "copy" else cp[1] > 0
    rule_rows.append({"run": run, "role": ROLE.get(src, "generator"), "pairs": n_judged, "miou": round(OBSERVED[run], 4),
                      "bar": BAR, "test1_reaches_bar": t1,
                      "minus_r9": r9[0], "minus_r9_ci95_low": r9[1], "minus_r9_ci95_high": r9[2], "test2_above_r9": t2,
                      "minus_copy": cp[0], "minus_copy_ci95_low": cp[1], "minus_copy_ci95_high": cp[2],
                      "test3_above_copy": t3, "passes_rule": bool(t1 and t2 and t3)})
rule = pd.DataFrame(rule_rows)
rule.to_csv(OUT_DIR / "rule_tests.csv", index=False)

# ── The five checks, and summary.json ──
n_rows = len(val_pairs) * 2 * (len(BEFORE_SOURCE) + 1)
in_smoke = " (smoke test: its 8 pairs, within 0.005)" if SMOKE_TEST else " (within 0.001)"

def repro_text(run):
    return "; ".join(f"{x.pairs} pairs: {x.miou} / {x.destroyed_iou}, E5 {x.e5_miou} / {x.e5_destroyed_iou}"
                     for x in repro[repro["run"] == run].itertuples())

CHECKS = [
    ("1. Step 0's pairs.csv: its full run, the same split, 605 clean", SAME_AS_STEP0,
     "; ".join(k for k, v in STEP0_FINDINGS.items() if not v) or "6,443 pairs, 5,798 / 645, 605 clean, same events"),
    ("2. The gray colour is the training before-photos' mean, (86, 91, 69)", GRAY_OK,
     f"{GRAY_RGB} from the mean {GRAY_MEAN} of {GRAY_PHOTOS:,} photos"),
    ("3. R11 real reproduces E5 step 2" + in_smoke, bool(repro.loc[repro["run"] == "R11 real", "reproduces_e5"].all()),
     repro_text("R11 real")),
    ("4. R9 reproduces E5 step 2" + in_smoke, bool(repro.loc[repro["run"] == "R9", "reproduces_e5"].all()),
     repro_text("R9")),
    ("5. Every run scored every validation pair",
     len(per_pair) == n_rows and bool(per_pair.groupby(["run", "mode"]).size().eq(len(val_pairs)).all()),
     f"{len(per_pair):,} rows: {len(RUNS)} runs x 2 modes x {len(val_pairs)} pairs"),
]
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "segmentation-models-pytorch": smp.__version__,
            "numpy": np.__version__, "opencv": cv2.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
summary = {
    "task": "Layer 3 step 1", "smoke_test": SMOKE_TEST, "minutes": round((time.time() - T_START) / 60, 1),
    "model_minutes": model_minutes, "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "versions": VERSIONS, "before_source": BEFORE_SOURCE,
    "checkpoints": {m: {"file": str(CKPTS[m]), "sha256": CKPT_SHA256[m]} for m in RUN_MODELS},
    "gray": {"rgb_uint8": GRAY_RGB, "mean": GRAY_MEAN, "sums": GRAY_SUMS, "pixels": GRAY_PIXELS, "photos": GRAY_PHOTOS},
    "step0_pairs_csv": str(STEP0_DIR / "pairs.csv"), "validation_pairs": len(val_pairs), "clean_pairs": n_clean,
    "reproduction": repro.to_dict("records"), "own_bar": round(OWN_BAR, 6), "bar": BAR,
    "resampling": {"resamples": RESAMPLES, "seed": SEED, "pairs": n_judged,
                   "method": "E7 step 2b's: multinomial weights over the pairs, percentiles 2.5 and 97.5"},
    "rule_tests": rule.to_dict("records"),
    "checks": {name: ("passed" if ok else "FAILED") for name, ok, _ in CHECKS},
}
(OUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)), encoding="utf-8")

# ── The printout ──
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
on = "the smoke test's 8 pairs" if SMOKE_TEST else "the 605 clean pairs"
print(f"Layer 3 step 1{' [SMOKE TEST]' if SMOKE_TEST else ''}: R11 real and R9 against E5 step 2 (8-view TTA)")
print(repro.to_string(index=False))
main = scores[(scores["mode"] == "tta") & (scores["pairs"] == ("all" if SMOKE_TEST else "clean")) & (scores["group"] == "all")]
print(f"\nEach run on {on}, 8-view TTA:")
print(main[["run", "n_pairs", "miou", "destroyed_iou", "building_iou", "building_precision"]].round(4).to_string(index=False))
if BAR is not None:
    print(f"\nThe bar for step 2: {BAR} (from our own numbers: {OWN_BAR:.4f})")
else:
    print(f"\nThe bar for step 2: not set ({'smoke test' if SMOKE_TEST else 'R11 real or R9 does not reproduce E5'}); "
          f"from these numbers it would be {OWN_BAR:.4f}")
print(f"\nThe rule's three tests on {on} (for real, copy and gray they decide nothing):")
print(rule[["run", "miou", "test1_reaches_bar", "minus_r9", "minus_r9_ci95_low", "minus_r9_ci95_high", "test2_above_r9",
            "minus_copy", "minus_copy_ci95_low", "minus_copy_ci95_high", "test3_above_copy", "passes_rule"]].to_string(index=False))
print(f"\nLayer 3 step 1{' [SMOKE TEST]' if SMOKE_TEST else ''}: the checks")
for name, ok, found in CHECKS:
    print(f"  {'✓' if ok else '✗'} {name}: {'passed' if ok else 'FAILED'} ({found})")
print(f"\n✓ Wrote {OUT_DIR}: per_pair.csv, scores.csv, rule_tests.csv, summary.json ({summary['minutes']} min in all)")
failed = [name for name, ok, _ in CHECKS if not ok]
if failed:
    raise RuntimeError("STOP: send this printout to Abrar. Failed: " + "; ".join(failed))
print("✓ All five checks passed" + (" on the smoke test's 8 pairs; the full run decides" if SMOKE_TEST else ""))

Layer 3 step 1: R11 real and R9 against E5 step 2 (8-view TTA)
     run pairs   miou  destroyed_iou  e5_miou  e5_destroyed_iou  tolerance  reproduces_e5
R11 real   605 0.5805         0.2692   0.5805            0.2692      0.001           True
R11 real   645 0.5875         0.2965   0.5875            0.2965      0.001           True
      R9   605 0.5402         0.2874   0.5402            0.2874      0.001           True
      R9   645 0.5440         0.3001   0.5440            0.3001      0.001           True

Each run on the 605 clean pairs, 8-view TTA:
     run  n_pairs   miou  destroyed_iou  building_iou  building_precision
R11 real      605 0.5805         0.2692        0.7429              0.8347
R11 copy      605 0.4046         0.0431        0.5040              0.6781
R11 gray      605 0.2464         0.0020        0.0390              0.8844
      R9      605 0.5402         0.2874        0.5905              0.6686

The bar for step 2: 0.5604 (from our own numbers: 0.5604)

The rule's 